# Solar Correlation Analysis

Explore how solar flux (SFI) affects HF propagation across different bands.

**What you'll learn:**
- The relationship between SFI and propagation quality
- How different bands respond to solar activity
- The Band-SFI curve: why high SFI helps high bands but hurts low bands

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ionis_jupyter import load_dataset, plot_solar_correlation

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load propagation data
df = load_dataset("wspr")  # Has avg_sfi column
print(f"Loaded {len(df):,} signatures")
print(f"SFI range: {df['avg_sfi'].min():.0f} - {df['avg_sfi'].max():.0f}")

In [ ]:
# Load solar indices for reference
try:
    solar = load_dataset("solar")
    print(f"Solar data: {len(solar):,} days")
    print(f"Date range: {solar['date'].min()} to {solar['date'].max()}")
except FileNotFoundError:
    print("Solar dataset not downloaded. Using propagation data SFI.")

## Solar Flux Index (SFI)

**What is SFI?**
- 10.7 cm radio flux from the Sun, measured in Solar Flux Units (SFU)
- Proxy for UV/EUV that ionizes the F-layer
- Range: ~65 (solar minimum) to ~300+ (solar maximum)

**Effect on propagation:**
- Higher SFI → more F-layer ionization → higher MUF
- High bands (10m, 15m) need high SFI to open
- Low bands (80m, 160m) suffer from D-layer absorption at high SFI

In [ ]:
# SFI correlation for a single band
plot_solar_correlation(df, band=111)  # 10m
plt.title("10m: SNR vs Solar Flux Index")
plt.show()

In [ ]:
# Compare multiple bands
BANDS = [103, 105, 107, 109, 111]  # 80m, 40m, 20m, 15m, 10m
BAND_NAMES = {103: "80m", 105: "40m", 107: "20m", 109: "15m", 111: "10m"}

fig, ax = plt.subplots(figsize=(12, 7))

for band in BANDS:
    band_df = df[df["band"] == band].copy()
    band_df["sfi_bracket"] = (band_df["avg_sfi"] // 20) * 20
    
    agg = band_df.groupby("sfi_bracket")["median_snr"].mean()
    ax.plot(agg.index, agg.values, marker="o", label=BAND_NAMES[band], linewidth=2)

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Solar Flux Index (SFU)")
ax.set_ylabel("Mean SNR (dB)")
ax.set_title("The Band-SFI Curve: How Bands Respond to Solar Activity")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## The Band-SFI Curve

**Key insight from IONIS research:**

The effect of SFI on propagation varies continuously across frequency:

- **Above ~10 MHz (30m):** Higher SFI = better propagation (F-layer ionization helps)
- **Below ~10 MHz (30m):** Higher SFI = worse propagation (D-layer absorption hurts)
- **Zero crossing at ~10 MHz:** 30m band is relatively SFI-neutral

This is why a simple "high SFI = good conditions" heuristic fails on 80m and 160m.

In [ ]:
# Geomagnetic correlation (Kp index)
# Higher Kp = more geomagnetic disturbance = worse HF conditions

fig, ax = plt.subplots(figsize=(10, 6))

kp_agg = df.groupby(df["avg_kp"].round())["median_snr"].mean()

ax.bar(kp_agg.index, kp_agg.values, color="coral", alpha=0.7, edgecolor="black")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Kp Index (rounded)")
ax.set_ylabel("Mean SNR (dB)")
ax.set_title("Geomagnetic Effect: SNR vs Kp Index")
ax.set_xticks(range(10))
plt.tight_layout()
plt.show()